#  Visualizing Thread Scheduling


Enter your name and student ID.

 * Name:
 * Student ID:


# 1. Introduction
* We have learned how to write a loop that runs on a GPU, in CUDA and in OpenMP
* What we have _not_ seen is what actually happens when it runs: which thread executes which iteration, when it starts, and how long it takes
* In this notebook we record a timestamp inside every iteration and draw a picture of it
* We do the same experiment twice, once in CUDA and once in OpenMP, so that you can see they behave the same way
* The questions we want to answer are
  * how long does a single iteration take?
  * what happens when we go from 1 thread to 32 threads (a warp)?
  * what happens when we keep adding threads inside a single thread block?
  * and when we add thread blocks?

# 2. How to work on this notebook
* Read text, modify cells if necessary, and execute them
* Many cells can be executed without modifying them
* Cells that need your work are typically preceded with <font color="green">text colored green explaining what you need to do</font>
* Problems with non-trivial work have a section header starting with <font color="green">"Problem X :"</font>

# 3. Jupyter AI
* You can ask Jupyter AI when you need help
* Open [pd06_visualize.chat](pd06_visualize.chat) in this folder
* Choose `OpenCode` in Persona Chooser
* You can try several models, but `LiteLLM (taulec)/GPT-5.3 Codex (UTokyo Azure)` is recommended for now
* It is instructed to help you but not to give you the answer directly to exercise problems; do not try to circumvent it

# 4. Setting up compilers
- The setup is the same as for the OpenMP we used previously for the CPU


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH

# 5. The experiment
* The loop we are going to watch is this one; in OpenMP,
```
#pragma omp target teams distribute parallel for num_teams(n_teams) num_threads(n_threads)
  for (long i = 0; i < L; i++) {
    iter_fun(a, b, i, M, N, R, T);
  }
```
and in CUDA,
```
__global__ void sched_rec_kernel(double a, double b, long L, long M, long N,
                                 record_t * R, unsigned long long * T) {
  long i = get_thread_id_x();   /* blockDim.x * blockIdx.x + threadIdx.x */
  if (i < L) iter_fun(a, b, i, M, N, R, T);
}
...
  sched_rec_kernel<<<n_blocks, n_threads>>>(a, b, L, M, N, R, T);
```
* The two are meant to be the same thing; OpenMP's $i$-th iteration and CUDA's $i$-th thread do the same work, and a _team_ is a _thread block_
* `iter_fun(a, b, i, M, N, R, T)` repeats `x = a * x + b` $M \times N$ times and reads the clock every $N$ of them, so each iteration leaves $M$ timestamps behind
```
void iter_fun(double a, double b, long i, long M, long N) {
  double x = ... i ...;
  for (long j = 0; j < M; j++) {
    record_clock();
    for (long k = 0; k < N; k++) {
      x = a * x + b;
    }
  }
  record_x(x);
}
```


## 5-1. Reading the GPU's clocks and registers
* To see when something happened we need a clock _on the GPU_, and to see where it happened we need the GPU's own registers
* `gpu_ptx.h` wraps them in ordinary-looking functions; it works both from CUDA (`nvcc`) and from OpenMP (`clang++`)

 |function           |CUDA equivalent|meaning                              |
 |-------------------|---------------|-------------------------------------|
 |`gpu_block_idx()`  |`blockIdx.x`   |which thread block am I in           |
 |`gpu_grid_dim()`   |`gridDim.x`    |how many thread blocks were launched |
 |`gpu_thread_idx()` |`threadIdx.x`  |which thread of the block am I       |
 |`gpu_block_dim()`  |`blockDim.x`   |how many threads my block really has |
 |`gpu_warp_id()`    |`%warpid`      |the physical warp slot I occupy      |
 |`gpu_lane_id()`    |`%laneid`      |which lane of the warp am I          |
 |`gpu_sm_id()`      |`%smid`        |which SM am I running on             |
 |`gpu_nsm()`        |`%nsmid`       |how many SMs this GPU has            |
 |`gpu_clock()`      |`clock64()`    |this SM's cycle counter              |
 |`gpu_globaltimer()`|---            |a device-wide nanosecond timer       |

* <font color=red>The two clocks are different animals, and the difference matters</font>
  * `gpu_clock()` counts cycles of the SM you happen to be running on.  It is cheap to read and precise, but two SMs' values cannot be compared
  * `gpu_globaltimer()` is a single counter shared by the whole device, so values taken on different SMs _can_ be compared, but reading it costs far more
  * So we read `gpu_clock()` inside the loop, and `gpu_globaltimer()` only once at the start and once at the end of each iteration
* Note: the inline PTX inside these functions cannot be compiled for the CPU, so it is guarded by a macro that only the device pass of the compiler defines (`__CUDA_ARCH__` for `nvcc`, `__NVPTX__` for `clang++`).  <font color=red>`nvc++` defines no such macro</font>, so use `nvcc` or `clang++` for this notebook


In [ ]:
%%writefile gpu_ptx.h
/*** gpu_ptx.h
     Read NVIDIA GPU special registers from inside a GPU kernel, whether the
     kernel was written in CUDA (compiled by nvcc) or in OpenMP (compiled by
     clang++ or nvc++).  They let you see for yourself which thread block,
     which warp and which SM a piece of work ended up on, and when it ran.

       gpu_block_idx()   blockIdx.x    which thread block am I in
       gpu_grid_dim()    gridDim.x     how many thread blocks were launched
       gpu_thread_idx()  threadIdx.x   which thread of the block am I
       gpu_block_dim()   blockDim.x    how many threads my block really has
       gpu_warp_id()     %warpid       the physical warp slot I occupy
                                       (NOT threadIdx.x / 32; see below)
       gpu_lane_id()     %laneid       which lane of the warp am I
       gpu_sm_id()       %smid         which SM am I running on
       gpu_nsm()         %nsmid        how many SMs this GPU has
       gpu_clock()       %clock64      this SM's cycle counter
       gpu_globaltimer() %globaltimer  a device-wide nanosecond timer

     The two clocks are different animals and the difference matters.
     %clock64 counts cycles of the SM you happen to be on, so it is precise
     but two SMs' values cannot be compared.  %globaltimer is a single
     counter shared by the whole device, so values from different SMs are
     comparable, but reading it costs much more than reading %clock64.
     Read %clock64 in an inner loop; read %globaltimer once at the start and
     once at the end.

     The inline PTX cannot be compiled for the host, so it is guarded by a
     macro that only the device pass of the compiler defines.  nvcc defines
     __CUDA_ARCH__ and clang++ defines __NVPTX__, so both work.  nvc++
     preprocesses the file only once and defines no such macro, so under
     nvc++ every function here returns -1; use nvcc or clang++.          ***/
#pragma once

#if defined(__CUDA_ARCH__) || defined(__NVPTX__)
#define GPU_DEVICE_PASS 1
#else
#define GPU_DEVICE_PASS 0
#endif

#if defined(__CUDACC__)
#define GPU_FN __device__ static inline
#else
#define GPU_FN static inline
#endif

#if GPU_DEVICE_PASS
#define GPU_SREG32(reg) ({ unsigned r_; asm volatile("mov.u32 %0, %%" reg ";" : "=r"(r_)); r_; })
#define GPU_SREG64(reg) ({ unsigned long long r_; asm volatile("mov.u64 %0, %%" reg ";" : "=l"(r_)); r_; })
#else
#define GPU_SREG32(reg) 0xffffffffu
#define GPU_SREG64(reg) 0xffffffffffffffffull
#endif

#if defined(_OPENMP)
#pragma omp declare target
#endif

GPU_FN unsigned gpu_block_idx(void)  { return GPU_SREG32("ctaid.x");  }
GPU_FN unsigned gpu_grid_dim(void)   { return GPU_SREG32("nctaid.x"); }
GPU_FN unsigned gpu_thread_idx(void) { return GPU_SREG32("tid.x");    }
GPU_FN unsigned gpu_block_dim(void)  { return GPU_SREG32("ntid.x");   }
GPU_FN unsigned gpu_warp_id(void)    { return GPU_SREG32("warpid");   }
GPU_FN unsigned gpu_lane_id(void)    { return GPU_SREG32("laneid");   }
GPU_FN unsigned gpu_sm_id(void)      { return GPU_SREG32("smid");     }
GPU_FN unsigned gpu_nsm(void)        { return GPU_SREG32("nsmid");    }

/* this SM's cycle counter; precise, but not comparable across SMs */
GPU_FN unsigned long long gpu_clock(void)       { return GPU_SREG64("clock64");     }
/* a device-wide nanosecond timer; comparable across SMs, but slow to read */
GPU_FN unsigned long long gpu_globaltimer(void) { return GPU_SREG64("globaltimer"); }

#if defined(_OPENMP)
#pragma omp end declare target
#endif

## 5-2. The part that is common to both versions

In [ ]:
%%writefile gpu_sched_rec.h
/*** gpu_sched_rec.h
     The part of the scheduling experiment that is the same whether the
     kernel is written in CUDA or in OpenMP.

     Each iteration i of the loop calls iter_fun(), which repeats
     x = a * x + b  (M * N) times, reading the SM cycle counter every N
     of them.  It also records, once, where it ran (SM, block, warp, lane)
     and a device-wide timestamp at its start and at its end.          ***/
#pragma once

#include <stdio.h>
#include "gpu_ptx.h"

typedef struct {
  double x;                     /* the (meaningless) result, so that the
                                   compiler cannot delete the loop */
  unsigned sm, nsm, block, thread, warp, lane;
  unsigned long long gt0, gt1;  /* %globaltimer at start / at end */
  unsigned long long c0, c1;    /* %clock64 read at the same two points,
                                   so that the two clocks can be related */
} record_t;

/* one iteration of the loop; T[i*M] .. T[i*M+M-1] receive the clocks */
GPU_FN void iter_fun(double a, double b, long i, long M, long N,
                     record_t * R, unsigned long long * T) {
  double x = i + 1;
  T = &T[i * M];
  unsigned sm = gpu_sm_id(), nsm = gpu_nsm();
  unsigned blk = gpu_block_idx(), thr = gpu_thread_idx();
  unsigned wrp = gpu_warp_id(), lne = gpu_lane_id();
  /* read the two clocks back to back into registers.  Do NOT put a store
     between them: the store can take hundreds of cycles and a different
     number of them in each thread, and the two timestamps would then no
     longer refer to the same moment. */
  unsigned long long g0 = gpu_globaltimer(), k0 = gpu_clock();
  for (long j = 0; j < M; j++) {
    T[j] = gpu_clock();
    for (long k = 0; k < N; k++) {
      x = a * x + b;
    }
  }
  unsigned long long k1 = gpu_clock(), g1 = gpu_globaltimer();
  R[i].sm = sm; R[i].nsm = nsm; R[i].block = blk; R[i].thread = thr;
  R[i].warp = wrp; R[i].lane = lne;
  R[i].gt0 = g0; R[i].c0 = k0; R[i].gt1 = g1; R[i].c1 = k1;
  R[i].x = x;
}

/* print the records in the format gpu_sched_vis.py understands */
static void dump(record_t * R, unsigned long long * T,
                 long L, long M, long N) {
  long k = 0;
  printf("#params L=%ld M=%ld N=%ld\n", L, M, N);
  for (long i = 0; i < L; i++) {
    printf("i=%ld sm=%u/%u block=%u thread=%u warp=%u lane=%u"
           " gt0=%llu gt1=%llu c0=%llu c1=%llu x=%f T=",
           i, R[i].sm, R[i].nsm, R[i].block, R[i].thread, R[i].warp, R[i].lane,
           R[i].gt0, R[i].gt1, R[i].c0, R[i].c1, R[i].x);
    for (long j = 0; j < M; j++) {
      printf("%s%llu", (j ? " " : ""), T[k]);
      k++;
    }
    printf("\n");
  }
}

## 5-3. The CUDA version
* It uses `cuda_util.h`, the small set of wrappers (`check_api_error`, `check_launch_error`, `dev_malloc`, `to_dev`, `to_host`, ...) you already used in the CUDA notebook


In [ ]:
%%writefile gpu_sched_rec.cu
/*** The same experiment as gpu_sched_rec.cc, written in CUDA.
     Build:  nvcc -arch=sm_90 -O3 gpu_sched_rec.cu -o cuda_sched_rec
     Run:    ./cuda_sched_rec N_BLOCKS N_THREADS_PER_BLOCK [L [M [N]]]  ***/
#include <stdio.h>
#include <stdlib.h>
#include "cuda_util.h"
#include "gpu_sched_rec.h"

__global__ void sched_rec_kernel(double a, double b, long L, long M, long N,
                                 record_t * R, unsigned long long * T) {
  long i = get_thread_id_x();
  if (i < L) iter_fun(a, b, i, M, N, R, T);
}


/* write to every byte we are going to record into, so that the pages are
   already on the GPU when the real kernel starts.  Without this, the first
   thread to touch a page stalls for ~200,000 cycles while the page is
   migrated, and that stall is what you would end up measuring. */
__global__ void warm_up(long L, long M, record_t * R, unsigned long long * T) {
  long i = get_thread_id_x(), n = get_nthreads_x();
  for (long k = i; k < L; k += n) R[k].sm = 0;
  for (long k = i; k < L * M; k += n) T[k] = 0;
}

int main(int argc, char ** argv) {
  int idx = 1;
  long n_blocks  = (idx < argc ? atol(argv[idx]) : 1);   idx++;
  long n_threads = (idx < argc ? atol(argv[idx]) : 32);  idx++;
  long L = (idx < argc ? atol(argv[idx]) : n_blocks * n_threads); idx++;
  long M = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  long N = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  double a = 0.99, b = 1.00;
  if (L != n_blocks * n_threads) {
    fprintf(stderr,
            "warning: L=%ld, but n_blocks x n_threads = %ld; only %ld of the"
            " %ld blocks do any work, the rest return immediately\n",
            L, n_blocks * n_threads, (L + n_threads - 1) / n_threads, n_blocks);
  }
  /* one array, visible from both the CPU and the GPU; no copying */
  record_t * R;
  unsigned long long * T;
  check_api_error(cudaMallocManaged(&R, L * sizeof(record_t)));
  check_api_error(cudaMallocManaged(&T, L * M * sizeof(unsigned long long)));

  check_launch_error((warm_up<<<n_blocks,n_threads>>>(L, M, R, T)));
  check_api_error(cudaDeviceSynchronize());

  struct cudaDeviceProp prop[1];
  check_api_error(cudaGetDeviceProperties(prop, 0));
  fprintf(stderr,
          "CUDA: <<<%ld,%ld>>> L=%ld M=%ld N=%ld (%d SMs, max clock %d kHz)\n",
          n_blocks, n_threads, L, M, N, prop->multiProcessorCount, get_freq());
  check_launch_error((sched_rec_kernel<<<n_blocks,n_threads>>>
                      (a, b, L, M, N, R, T)));
  check_api_error(cudaDeviceSynchronize());

  dump(R, T, L, M, N);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvcc -arch=sm_90 -O3 gpu_sched_rec.cu -o cuda_sched_rec

## 5-4. The OpenMP version

In [ ]:
%%writefile gpu_sched_rec.cc
/*** The same experiment as gpu_sched_rec.cu, written in OpenMP.
     Build:  clang++ -Wall -fopenmp --offload-arch=sm_90 -O3 \
                     gpu_sched_rec.cc -o omp_sched_rec
     Run:    ./omp_sched_rec N_TEAMS N_THREADS_PER_TEAM [L [M [N]]]     ***/
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
#include "gpu_sched_rec.h"

/* CPU and GPU share memory, so no map clause is needed anywhere below */
#pragma omp requires unified_shared_memory


int main(int argc, char ** argv) {
  int idx = 1;
  long n_teams   = (idx < argc ? atol(argv[idx]) : 1);   idx++;
  long n_threads = (idx < argc ? atol(argv[idx]) : 32);  idx++;
  long L = (idx < argc ? atol(argv[idx]) : n_teams * n_threads); idx++;
  long M = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  long N = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  double a = 0.99, b = 1.00;
  if (L != n_teams * n_threads) {
    fprintf(stderr,
            "warning: L=%ld, but n_teams x n_threads = %ld; only %ld of the"
            " %ld teams do any work, the rest return immediately\n",
            L, n_teams * n_threads, (L + n_threads - 1) / n_threads, n_teams);
  }
  record_t * R = (record_t *)calloc(L, sizeof(record_t));
  unsigned long long * T
    = (unsigned long long *)calloc(L * M, sizeof(unsigned long long));

  /* touch everything from the GPU first, so that the pages are already
     there when the real kernel starts; otherwise the first thread to reach
     a page stalls while it is moved, and that stall is what you measure */
#pragma omp target teams distribute parallel for \
  num_teams(n_teams) num_threads(n_threads)
  for (long k = 0; k < L * M; k++) {
    T[k] = 0;
    if (k < L) R[k].sm = 0;
  }

  fprintf(stderr, "OpenMP: num_teams(%ld) num_threads(%ld) L=%ld M=%ld N=%ld\n",
          n_teams, n_threads, L, M, N);
#pragma omp target teams distribute parallel for \
  num_teams(n_teams) num_threads(n_threads)
  for (long i = 0; i < L; i++) {
    iter_fun(a, b, i, M, N, R, T);
  }
  dump(R, T, L, M, N);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall -fopenmp --offload-arch=sm_90 -O3 gpu_sched_rec.cc -o omp_sched_rec

* Neither program copies anything between CPU and GPU: the CUDA one allocates with `cudaMallocManaged`, and the OpenMP one declares `#pragma omp requires unified_shared_memory` and uses no `map` clause at all
* Each of them does, however, run a short <font color=blue>warm-up kernel</font> first that writes a zero to every element it is going to record into.  <font color=red>Without it the measurement is destroyed</font>: the first thread to touch a page stalls for about 200,000 cycles while the page is brought to the GPU, and that stall is what you would see in the picture.  Touching the memory once beforehand is enough to get rid of it
* Both programs take the same arguments
```
./cuda_sched_rec N_BLOCKS N_THREADS_PER_BLOCK [L [M [N]]]
./omp_sched_rec  N_TEAMS  N_THREADS_PER_TEAM  [L [M [N]]]
```
and print the same thing, one line per iteration, so the same script draws both

# 6. 1-thread execution
* Let's start from the simplest possible thing: a single thread executing a single iteration
* $M = 10$ timestamps are taken, every $N = 10000$ repetitions of `x = a * x + b`
* <font color=green>Exercise:</font> From the plot, calculate how many cycles a single multiply-add takes; `summary()` divides it by $N$ and prints the cycles for one `x = a * x + b` on the last line
* You should get about <font color=blue>8.4 cycles</font>, and that is a _latency_: each `x = a * x + b` has to wait for the previous one, because they all update the same `x`
* Note: 
  * What we observe is a _latency_, or the number of cycles between this instruction and the instruction that uses its result
  * This should be distinguished from _throughput_, the rate at which this GPU can retire multiply-adds, which is much higher than what the latency suggests (1/8)


In [ ]:
%%sub
T=1
H=1
M=10
N=10000
L=$((T * H))
./cuda_sched_rec ${T} ${H} ${L} ${M} ${N} > c${T}x${H}.dat
./omp_sched_rec  ${T} ${H} ${L} ${M} ${N} > o${T}x${H}.dat

In [ ]:
import importlib, gpu_sched_vis
importlib.reload(gpu_sched_vis)
print("=== CUDA 1 thread block x 1 thread ===")
gpu_sched_vis.summary("c1x1.dat")
print("=== OpenMP 1 team x 1 thread ===")
gpu_sched_vis.summary("o1x1.dat")
print("=== CUDA 1 thread block x 1 thread ===")
gpu_sched_vis.sched_plt("c1x1.dat", color_by="thread")
print("=== OpenMP 1 team x 1 thread ===")
gpu_sched_vis.sched_plt("o1x1.dat", color_by="thread")

* Observe:
  * The _latency_ of a single double precision FMA (`x = a * x + b`), which is approximately 8 cycles
  * In other words, the _throughput_ of one thread is 1/8 FMA per cycle
  * This is obviously far from the maximum throughput (64 double precision FMAs per SM per cycle)

# 7. 1-warp execution
* Now 32 threads, which is exactly one warp
* <font color=green>Exercise:</font> Before running it, predict how long it will take compared to the single thread.  Then look at the picture


In [ ]:
%%sub
T=1
H=32
M=10
N=10000
L=$((T * H))
./cuda_sched_rec ${T} ${H} ${L} ${M} ${N} > c${T}x${H}.dat
./omp_sched_rec  ${T} ${H} ${L} ${M} ${N} > o${T}x${H}.dat

In [ ]:
import importlib, gpu_sched_vis
importlib.reload(gpu_sched_vis)
print("=== CUDA 1 thread block x 32 threads ===")
gpu_sched_vis.summary("c1x32.dat")
print("=== OpenMP 1 team x 32 threads ===")
gpu_sched_vis.summary("o1x32.dat")
print("=== CUDA 1 thread block x 32 threads ===")
gpu_sched_vis.sched_plt("c1x32.dat", color_by="lane")
print("=== OpenMP 1 team x 32 threads ===")
gpu_sched_vis.sched_plt("o1x32.dat", color_by="lane")

* Observe:
  * The dots form <font color=blue>perfectly vertical lines</font>: all 32 threads of the warp take their $j$-th timestamp at _exactly_ the same cycle
  * That is, threads in a warp execute the same instruction in the same cycle
  * You can confirm that the clock values are identical by looking at the `.dat` file
  * The latency does not increase at all, so, in terms of throughput, it becomes almost 32 times as high, or about 4 FMAs per cycle (1/8 $times$ 32)
  * The OpenMP version gives the same picture; `teams`/`distribute`/`parallel`/`for` end up producing exactly the same CUDA threads

# 8. (Aside) warp divergence
* What happens when threads in a single warp do not follow the same control path (e.g., `if` statement branches differently between threads)?
* The GPU hardware <font color="red">cannot let different threads in the same warp execute different instructions in the same cycle</font>
* The hardware runs <font color=red>one branch after the other</font>, with the threads that did not choose it switched off
* This is called <font color=blue>warp divergence</font>
* To see it, we replace `iter_fun()` with `iter_fun_div()`, in which thread $t$ takes branch $(t / G) \bmod D$ of a $D$-way branch
  * $G = 1$ : neighbouring threads take different branches, so the branches diverge _inside_ a warp
  * $G = 32$ : a whole warp takes the same branch, so nothing diverges
* Every thread does the same amount of work in either case; only _who_ takes _which_ branch changes


In [ ]:
%%writefile gpu_div_rec.h
/*** gpu_div_rec.h
     A variation of iter_fun() used only for the warp divergence detour.

     Threads of the same warp no longer all run the same code: thread t
     takes branch  (t / G) % D  of a D-way branch, and only the threads
     that took the same branch can run together.

       G = 1  : neighbouring threads take different branches, so the
                branches are divergent *inside* a warp
       G = 32 : a whole warp takes the same branch, so nothing diverges

     Everything else (what is measured and how it is printed) is exactly
     the same as in gpu_sched_rec.h.                                    ***/
#pragma once

#include "gpu_sched_rec.h"

/* The D-way branch is written as a loop over the branches, with the body
   guarded by "is this my branch?".  That is precisely what the hardware
   does with an if / else-if / else chain anyway: it walks the branches one
   after another, running each one with only the threads that chose it
   switched on.  Writing it this way lets us vary D at run time. */
GPU_FN void iter_fun_div(double a, double b, long i, long M, long N,
                         long D, long G,
                         record_t * R, unsigned long long * T) {
  double x = i + 1;
  T = &T[i * M];
  unsigned sm = gpu_sm_id(), nsm = gpu_nsm();
  unsigned blk = gpu_block_idx(), thr = gpu_thread_idx();
  unsigned wrp = gpu_warp_id(), lne = gpu_lane_id();
  long mine = (thr / G) % D;    /* the branch this thread takes */
  unsigned long long g0 = gpu_globaltimer(), k0 = gpu_clock();
  for (long d = 0; d < D; d++) {
    if (d == mine) {
      for (long j = 0; j < M; j++) {
        T[j] = gpu_clock();
        for (long k = 0; k < N; k++) {
          x = a * x + b;
        }
      }
    }
  }
  unsigned long long k1 = gpu_clock(), g1 = gpu_globaltimer();
  R[i].sm = sm; R[i].nsm = nsm; R[i].block = blk; R[i].thread = thr;
  R[i].warp = wrp; R[i].lane = lne;
  R[i].gt0 = g0; R[i].c0 = k0; R[i].gt1 = g1; R[i].c1 = k1;
  R[i].x = x;
}

In [ ]:
%%writefile gpu_div_rec.cu
/*** The warp divergence experiment, in CUDA.
     Build:  nvcc -arch=sm_90 -O3 gpu_div_rec.cu -o cuda_div_rec
     Run:    ./cuda_div_rec N_BLOCKS N_THREADS_PER_BLOCK [L [M [N [D [G]]]]]
     D = how many branches, G = how many consecutive threads share one  ***/
#include <stdio.h>
#include <stdlib.h>
#include "cuda_util.h"
#include "gpu_div_rec.h"

__global__ void div_rec_kernel(double a, double b, long L, long M, long N,
                               long D, long G,
                               record_t * R, unsigned long long * T) {
  long i = get_thread_id_x();
  if (i < L) iter_fun_div(a, b, i, M, N, D, G, R, T);
}

/* see gpu_sched_rec.cu: touch the pages before we start measuring */
__global__ void warm_up(long L, long M, record_t * R, unsigned long long * T) {
  long i = get_thread_id_x(), n = get_nthreads_x();
  for (long k = i; k < L; k += n) R[k].sm = 0;
  for (long k = i; k < L * M; k += n) T[k] = 0;
}

int main(int argc, char ** argv) {
  int idx = 1;
  long n_blocks  = (idx < argc ? atol(argv[idx]) : 1);   idx++;
  long n_threads = (idx < argc ? atol(argv[idx]) : 32);  idx++;
  long L = (idx < argc ? atol(argv[idx]) : n_blocks * n_threads); idx++;
  long M = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  long N = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  long D = (idx < argc ? atol(argv[idx]) : 2);    idx++;
  long G = (idx < argc ? atol(argv[idx]) : 1);    idx++;
  double a = 0.99, b = 1.00;
  if (L != n_blocks * n_threads) {
    fprintf(stderr,
            "warning: L=%ld, but n_blocks x n_threads = %ld; only %ld of the"
            " %ld blocks do any work, the rest return immediately\n",
            L, n_blocks * n_threads, (L + n_threads - 1) / n_threads, n_blocks);
  }
  record_t * R;
  unsigned long long * T;
  check_api_error(cudaMallocManaged(&R, L * sizeof(record_t)));
  check_api_error(cudaMallocManaged(&T, L * M * sizeof(unsigned long long)));

  check_launch_error((warm_up<<<n_blocks,n_threads>>>(L, M, R, T)));
  check_api_error(cudaDeviceSynchronize());

  struct cudaDeviceProp prop[1];
  check_api_error(cudaGetDeviceProperties(prop, 0));
  fprintf(stderr,
          "CUDA: <<<%ld,%ld>>> L=%ld M=%ld N=%ld D=%ld G=%ld"
          " (%d SMs, max clock %d kHz)\n",
          n_blocks, n_threads, L, M, N, D, G,
          prop->multiProcessorCount, get_freq());
  check_launch_error((div_rec_kernel<<<n_blocks,n_threads>>>
                      (a, b, L, M, N, D, G, R, T)));
  check_api_error(cudaDeviceSynchronize());

  dump(R, T, L, M, N);
  return 0;
}

In [ ]:
%%writefile gpu_div_rec.cc
/*** The warp divergence experiment, in OpenMP.
     Build:  clang++ -Wall -fopenmp --offload-arch=sm_90 -O3 \
                     gpu_div_rec.cc -o omp_div_rec
     Run:    ./omp_div_rec N_TEAMS N_THREADS_PER_TEAM [L [M [N [D [G]]]]]
     D = how many branches, G = how many consecutive threads share one  ***/
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
#include "gpu_div_rec.h"

#pragma omp requires unified_shared_memory

int main(int argc, char ** argv) {
  int idx = 1;
  long n_teams   = (idx < argc ? atol(argv[idx]) : 1);   idx++;
  long n_threads = (idx < argc ? atol(argv[idx]) : 32);  idx++;
  long L = (idx < argc ? atol(argv[idx]) : n_teams * n_threads); idx++;
  long M = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  long N = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  long D = (idx < argc ? atol(argv[idx]) : 2);    idx++;
  long G = (idx < argc ? atol(argv[idx]) : 1);    idx++;
  double a = 0.99, b = 1.00;
  if (L != n_teams * n_threads) {
    fprintf(stderr,
            "warning: L=%ld, but n_teams x n_threads = %ld; only %ld of the"
            " %ld teams do any work, the rest return immediately\n",
            L, n_teams * n_threads, (L + n_threads - 1) / n_threads, n_teams);
  }
  record_t * R = (record_t *)calloc(L, sizeof(record_t));
  unsigned long long * T
    = (unsigned long long *)calloc(L * M, sizeof(unsigned long long));

  /* see gpu_sched_rec.cc: touch the pages before we start measuring */
#pragma omp target teams distribute parallel for \
  num_teams(n_teams) num_threads(n_threads)
  for (long k = 0; k < L * M; k++) {
    T[k] = 0;
    if (k < L) R[k].sm = 0;
  }

  fprintf(stderr, "OpenMP: num_teams(%ld) num_threads(%ld)"
          " L=%ld M=%ld N=%ld D=%ld G=%ld\n",
          n_teams, n_threads, L, M, N, D, G);
#pragma omp target teams distribute parallel for \
  num_teams(n_teams) num_threads(n_threads)
  for (long i = 0; i < L; i++) {
    iter_fun_div(a, b, i, M, N, D, G, R, T);
  }
  dump(R, T, L, M, N);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
nvcc -arch=sm_90 -O3 gpu_div_rec.cu -o cuda_div_rec
clang++ -Wall -fopenmp --offload-arch=sm_90 -O3 gpu_div_rec.cc -o omp_div_rec

* The two programs take the same arguments as before, with $D$ and $G$ appended
```
./cuda_div_rec N_BLOCKS N_THREADS_PER_BLOCK L M N D G
./omp_div_rec  N_TEAMS  N_THREADS_PER_TEAM  L M N D G
```
and the data files are named `c{T}x{H}x{D}x{G}.dat` and `o{T}x{H}x{D}x{G}.dat`

## 8-1. Divergence inside a warp
* One warp ($H = 32$), splitting it into $D = 1, 2, 4, 8$ groups by lane number


In [ ]:
%%sub
T=1
H=32
M=10
N=10000
G=1
L=$((T * H))
for D in 1 2 4 8 ; do
	./cuda_div_rec ${T} ${H} ${L} ${M} ${N} ${D} ${G} > c${T}x${H}x${D}x${G}.dat
	./omp_div_rec  ${T} ${H} ${L} ${M} ${N} ${D} ${G} > o${T}x${H}x${D}x${G}.dat
done

In [ ]:
import importlib, gpu_sched_vis
importlib.reload(gpu_sched_vis)
T, H, G = 1, 32, 1
for D in [1, 2, 4, 8]:
    print(f"=== CUDA {T} thread block x {H} threads, {D}-way branch (G={G}) ===")
    gpu_sched_vis.summary(f"c{T}x{H}x{D}x{G}.dat")
    gpu_sched_vis.sched_plt(f"c{T}x{H}x{D}x{G}.dat", color_by="lane")
    print(f"=== OpenMP {T} team x {H} threads, {D}-way branch (G={G}) ===")
    gpu_sched_vis.summary(f"o{T}x{H}x{D}x{G}.dat")
    gpu_sched_vis.sched_plt(f"o{T}x{H}x{D}x{G}.dat", color_by="lane")

* Observe:
  * The picture breaks into <font color=blue>$D$ bands</font>, one per branch; inside a band only the lanes that chose that branch make progress, and the rest of the warp is idle
  * The wall time is almost exactly $D$ times the $D = 1$ time, and the throughput $1/D$ of it
  * The cycles for one `x = a * x + b` (the last line of `summary()`) does <font color=red>not</font> change.  Divergence does not make an instruction slower; it just runs the branches one after another instead of together

## 8-2. The same branch, taken per warp
* The issue is solely about different threads _in the same warp_ taking different branches
* Now let $H = 128$, i.e. 4 warps, and the same $D$-way branch, but with $G = 32$, so that each warp takes one branch as a whole
* <font color=green>Exercise:</font> Before running it, predict what changes


In [ ]:
%%sub
T=1
H=128
M=10
N=10000
D=4
L=$((T * H))
for G in 1 32 ; do
	./cuda_div_rec ${T} ${H} ${L} ${M} ${N} ${D} ${G} > c${T}x${H}x${D}x${G}.dat
	./omp_div_rec  ${T} ${H} ${L} ${M} ${N} ${D} ${G} > o${T}x${H}x${D}x${G}.dat
done

In [ ]:
import importlib, gpu_sched_vis
importlib.reload(gpu_sched_vis)
T, H, D = 1, 128, 4
for G in [1, 32]:
    print(f"=== CUDA {T} thread block x {H} threads, {D}-way branch (G={G}) ===")
    gpu_sched_vis.summary(f"c{T}x{H}x{D}x{G}.dat")
    gpu_sched_vis.sched_plt(f"c{T}x{H}x{D}x{G}.dat", color_by="warp")
    print(f"=== OpenMP {T} team x {H} threads, {D}-way branch (G={G}) ===")
    gpu_sched_vis.summary(f"o{T}x{H}x{D}x{G}.dat")
    gpu_sched_vis.sched_plt(f"o{T}x{H}x{D}x{G}.dat", color_by="warp")

* Observe:
  * With $G = 32$ the bands are gone and the time is back to what a branch-free run takes, although the program is the same and each thread still skips $D - 1$ branches
  * A branch costs nothing as long as all 32 threads of a warp agree on it; it costs a factor of $D$ when they do not
  * This is why "avoid divergence" in GPU programming means "make threads that are in the same warp take the same path", not "avoid `if`"

# 9. 1-thread-block execution
* Keep a single thread block and increase the number of threads in it: 64, 128, 256, 384, 512, 640, 768, 896, 1024 (the hardware maximum)
* Now there is more than one warp per block, and they have to share the SM


In [ ]:
%%sub
T=1
M=10
N=10000
for H in 64 128 256 384 512 640 768 896 1024 ; do
	L=$((T * H))
	./cuda_sched_rec ${T} ${H} ${L} ${M} ${N} > c${T}x${H}.dat
	./omp_sched_rec  ${T} ${H} ${L} ${M} ${N} > o${T}x${H}.dat
done

In [ ]:
import importlib, gpu_sched_vis
importlib.reload(gpu_sched_vis)
T = 1
for H in [64, 128, 256, 384, 512, 640, 768, 896, 1024]:
    print(f"=== CUDA {T} thread block x {H} threads ===")
    gpu_sched_vis.summary(f"c{T}x{H}.dat")
    print(f"=== OpenMP {T} team x {H} threads ===")
    gpu_sched_vis.summary(f"o{T}x{H}.dat")

In [ ]:
import gpu_sched_vis
T = 1
for H in [64, 128, 256, 384, 512, 640, 768, 896, 1024]:
    print(f"=== CUDA {T} thread block x {H} threads ===")
    gpu_sched_vis.sched_plt(f"c{T}x{H}.dat", color_by="warp")
    print(f"=== OpenMP {T} team x {H} threads ===")
    gpu_sched_vis.sched_plt(f"o{T}x{H}.dat", color_by="warp")

* The picture below shows how throughput scales with the number of threads
* <font color=green>Exercise: </font> predict the shape of the plot before you see it


In [ ]:
import importlib, gpu_sched_vis
importlib.reload(gpu_sched_vis)
T = 1
HS = [1, 32, 64, 128, 256, 384, 512, 640, 768, 896, 1024]   # 1 and 32 come from the two sections above
gpu_sched_vis.throughput_plt({
	"CUDA"   : [f"c{T}x{H}.dat" for H in HS],
	"OpenMP" : [f"o{T}x{H}.dat" for H in HS],
}, peak_sms=1)   # a single block can only ever use one SM

* Observe:
  * The throughput continues to increase almost linearly up to 512 threads/block, where it reaches about 55 FMAs / cycle
  * Note that we are still using a single block (i.e., single SM)
  * Past 512 threads it flattens out at about 57, and the maximum a single SM can do is 64 double precision FMAs per cycle, so we are at about 90% of it, just by increasing the thread count
  * The remaining gap narrows as you increase $N$, so part of it is the overhead of taking and recording the clocks

# 10. Multiple thread-block execution
* Now keep the block at its largest, $H = 1024$, so that a single block already brings its own SM close to the limit, and start adding blocks
* The number to beat is now <font color=red>(number of SMs) $\times$ 64 FMA / cycle</font>, so 2048 on a 32-SM partition and 1664 on a 26-SM one
* <font color=red>From here on you must use `clock="global"`</font>: different blocks run on different SMs, and each SM's `%clock64` starts from its own value, so the cycle counts are not comparable.  `%globaltimer` is shared by the whole device


In [ ]:
%%sub
H=1024
M=10
N=10000
for T in 1 2 4 8 16 24 32 40 48 56 64 80 96 112 128 ; do
	L=$((T * H))
	./cuda_sched_rec ${T} ${H} ${L} ${M} ${N} > c${T}x${H}.dat
	./omp_sched_rec  ${T} ${H} ${L} ${M} ${N} > o${T}x${H}.dat
done

In [ ]:
import importlib, gpu_sched_vis
importlib.reload(gpu_sched_vis)
H = 1024
for T in [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 80, 96, 112, 128]:
	print(f"=== CUDA {T} thread blocks x {H} threads ===")
	gpu_sched_vis.summary(f"c{T}x{H}.dat")
	print(f"=== OpenMP {T} teams x {H} threads ===")
	gpu_sched_vis.summary(f"o{T}x{H}.dat")

In [ ]:
import gpu_sched_vis
H = 1024
for T in [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 80, 96, 112, 128]:
	print(f"=== CUDA {T} thread blocks x {H} threads ===")
	gpu_sched_vis.sched_plt(f"c{T}x{H}.dat", color_by="sm", clock="global", show_every=32)
	print(f"=== OpenMP {T} teams x {H} threads ===")
	gpu_sched_vis.sched_plt(f"o{T}x{H}.dat", color_by="sm", clock="global", show_every=32)

* The picture below shows how throughput scales with the number of teams or thread blocks
* <font color=green>Exercise:</font> predict the shape of the plot before you see it


In [ ]:
import importlib, gpu_sched_vis
importlib.reload(gpu_sched_vis)
H = 1024
TS = [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 80, 96, 112, 128]
gpu_sched_vis.throughput_plt({
	"CUDA"   : [f"c{T}x{H}.dat" for T in TS],
	"OpenMP" : [f"o{T}x{H}.dat" for T in TS],
})

* Note that the number of SMs allocated to your job is 132 when you use the whole device (`lecture` queue) and either 32 or 26 when you use a Multi-Instance GPU (MIG) instance (`lecture-mig` queue)
* You cannot know in advance which one your job lands on
* Either way, the expectation is that the throughput scales up to the number of SMs, because with 1024 threads in each thread block, a single block already saturates a single SM

# <font color="green"> Problem 1 :  Wrap-up Quizes</font>
On this device (Hopper, H200 GPU)

1. What is the _latency_ of a single double-precision FMA, in terms of cycles?


double-click and write your answer here

2. What is the maximum _throughput_ of double-precision FMAs per SM, in terms of FMAs per cycle?


double-click and write your answer here

3. What is the maximum throughput of double-precision FMAs per device (assuming 132 SMs), in terms of operations per cycle?


double-click and write your answer here

4. Assuming processor frequency of 2.0 GHz, translate this number into Floating Point Operations Per Second (FLOPS) and compare it with the published information (Google H200 Datasheet)


double-click and write your answer here